In [1]:
# CELL 0: Verify GPU is active
!nvidia-smi

Thu Oct 22 17:29:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# ============================================================
# CELL 1: Install only what is needed (no 'datasets' library)
# tqdm is used for the progress bar during processing
# ============================================================
!pip install tqdm -q
print("✓ Dependencies installed.")

✓ Dependencies installed.


In [7]:
# ============================================================
# CELL 2: Mount Google Drive and create project directories
# ============================================================
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Define directory paths
BASE_DIR = "/content/drive/MyDrive/NLP_Assignment_2_Project7"
DATA_DIR = os.path.join(BASE_DIR, "data")
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
OUT_DIR  = os.path.join(BASE_DIR, "outputs")

# Create all directories
for d in [DATA_DIR, CKPT_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"✓ Directory ready: {d}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Directory ready: /content/drive/MyDrive/NLP_Assignment_2_Project7/data
✓ Directory ready: /content/drive/MyDrive/NLP_Assignment_2_Project7/checkpoints
✓ Directory ready: /content/drive/MyDrive/NLP_Assignment_2_Project7/outputs


In [9]:
# ============================================================
# CELL 3: Download HaluEval QA dataset
#
# FIX 1: Uses urllib (standard library) — no 'datasets' lib needed.
# FIX 2: The file is JSONL (one JSON object per line), NOT a
#         JSON array. Parsing it with json.loads() on the full
#         string causes "Extra data" error. We parse line-by-line.
#
# CHECKPOINT GUARD: Skips download if file already exists in Drive.
# ============================================================
import json
import os
import urllib.request

raw_path = os.path.join(DATA_DIR, "halueval_qa_raw.json")

# --- Checkpoint guard: skip download if already saved ---
if os.path.exists(raw_path):
    print(f"✓ Found existing dataset in Drive. Loading from cache...")
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"  Loaded {len(raw_data)} pairs from: {raw_path}")

else:
    # Download directly from the official RUCAIBox GitHub repo
    # Source: https://github.com/RUCAIBox/HaluEval (Li et al., 2023)
    url = "https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/qa_data.json"
    print(f"Downloading HaluEval QA dataset from GitHub...")
    print(f"  Source: {url}")

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        raw_text = response.read().decode("utf-8")

    # The file is JSONL (one JSON object per line), NOT a JSON array.
    # json.loads() on the full string raises "Extra data" — parse line by line.
    raw_json_data = []
    for line in raw_text.strip().splitlines():
        line = line.strip()
        if line:  # skip blank lines
            raw_json_data.append(json.loads(line))

    # Format into clean records with sequential integer IDs
    raw_data = []
    for idx, item in enumerate(raw_json_data):
        raw_data.append({
            "id": idx,
            "knowledge": item.get("knowledge", ""),
            "question": item["question"],
            "right_answer": item["right_answer"],
            "hallucinated_answer": item["hallucinated_answer"],
        })

    # Save to Google Drive (permanent storage)
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(raw_data, f, indent=2, ensure_ascii=False)
    print(f"✓ Dataset saved to Drive: {raw_path}")

print(f"\n  Total pairs: {len(raw_data)}")
print(f"\n--- Sample Record ---")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))


  Source: https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/qa_data.json
✓ Dataset saved to Drive: /content/drive/MyDrive/NLP_Assignment_2_Project7/data/halueval_qa_raw.json

  Total pairs: 10000

--- Sample Record ---
{
  "id": 0,
  "knowledge": "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.",
  "question": "Which magazine was started first Arthur's Magazine or First for Women?",
  "right_answer": "Arthur's Magazine",
  "hallucinated_answer": "First for Women was started first."
}


In [10]:
# ============================================================
# CELL 4: Draw a fixed random sample (tractability)
#
# Assignment instruction:
#   "draw a fixed random sample for tractability if needed"
#
# We sample 400 pairs = 800 scoring instances.
# CHECKPOINT GUARD: Skips re-sampling if already saved.
# ============================================================
import random

RANDOM_SEED = 42
SAMPLE_SIZE = 400  # 400 questions → 800 scoring instances

sampled_path = os.path.join(DATA_DIR, "halueval_qa_sampled.json")

if os.path.exists(sampled_path):
    print(f"✓ Found existing sample in Drive. Loading from cache...")
    with open(sampled_path, "r", encoding="utf-8") as f:
        sampled_data = json.load(f)
    print(f"  Loaded {len(sampled_data)} sampled pairs from: {sampled_path}")

else:
    random.seed(RANDOM_SEED)
    sampled_data = random.sample(raw_data, SAMPLE_SIZE)

    # Assign clean sequential sample IDs
    for i, item in enumerate(sampled_data):
        item["sample_id"] = i

    with open(sampled_path, "w", encoding="utf-8") as f:
        json.dump(sampled_data, f, indent=2, ensure_ascii=False)
    print(f"✓ Sampled subset saved to Drive: {sampled_path}")

print(f"  Sample size : {len(sampled_data)} pairs")
print(f"  Random seed : {RANDOM_SEED}")
print(f"  Scoring inst: {len(sampled_data) * 2} (2 per question)")

✓ Sampled subset saved to Drive: /content/drive/MyDrive/NLP_Assignment_2_Project7/data/halueval_qa_sampled.json
  Sample size : 400 pairs
  Random seed : 42
  Scoring inst: 800 (2 per question)


In [11]:
# ============================================================
# CELL 5: Build fixed 80/20 train/test partition
#
# Assignment instruction:
#   "build your own 80/20 train/test partition for Task 3"
#
# Split is at QUESTION level so both answer types for a
# question always stay in the same partition (no leakage).
# CHECKPOINT GUARD: Skips re-splitting if already saved.
# ============================================================

split_path = os.path.join(DATA_DIR, "train_test_split.json")

if os.path.exists(split_path):
    print(f"✓ Found existing split in Drive. Loading from cache...")
    with open(split_path, "r", encoding="utf-8") as f:
        split_info = json.load(f)
    train_ids = split_info["train_sample_ids"]
    test_ids  = split_info["test_sample_ids"]

else:
    random.seed(RANDOM_SEED)
    indices = list(range(len(sampled_data)))
    random.shuffle(indices)

    split_point = int(0.8 * len(sampled_data))  # 320 train, 80 test
    train_ids   = sorted(indices[:split_point])
    test_ids    = sorted(indices[split_point:])

    split_info = {
        "random_seed"           : RANDOM_SEED,
        "sample_size"           : len(sampled_data),
        "train_size"            : len(train_ids),
        "test_size"             : len(test_ids),
        "train_scoring_instances": len(train_ids) * 2,
        "test_scoring_instances" : len(test_ids) * 2,
        "train_sample_ids"      : train_ids,
        "test_sample_ids"       : test_ids,
    }

    with open(split_path, "w") as f:
        json.dump(split_info, f, indent=2)
    print(f"✓ Train/Test split saved to Drive: {split_path}")

print(f"  Train: {len(train_ids)} questions → {len(train_ids)*2} scoring instances")
print(f"  Test : {len(test_ids)} questions → {len(test_ids)*2} scoring instances")

✓ Train/Test split saved to Drive: /content/drive/MyDrive/NLP_Assignment_2_Project7/data/train_test_split.json
  Train: 320 questions → 640 scoring instances
  Test : 80 questions → 160 scoring instances


In [12]:
# ============================================================
# CELL 6: Expand pairs to individual scoring instances
#
# Assignment instruction:
#   "Treat each pair as two separate scoring instances
#    for AUC-PR purposes."
#
# Each question generates:
#   label=0  →  right_answer        (faithful)
#   label=1  →  hallucinated_answer (hallucinated)
# ============================================================

def expand_to_scoring_instances(data, id_list):
    instances = []
    for sample_id in id_list:
        item = data[sample_id]
        # Instance A — right answer (label 0: faithful)
        instances.append({
            "instance_id"     : f"{sample_id}_right",
            "sample_id"       : sample_id,
            "original_id"     : item["id"],
            "knowledge"       : item["knowledge"],
            "question"        : item["question"],
            "candidate_answer": item["right_answer"],
            "label"           : 0,
            "answer_type"     : "right_answer",
        })
        # Instance B — hallucinated answer (label 1: hallucinated)
        instances.append({
            "instance_id"     : f"{sample_id}_hallucinated",
            "sample_id"       : sample_id,
            "original_id"     : item["id"],
            "knowledge"       : item["knowledge"],
            "question"        : item["question"],
            "candidate_answer": item["hallucinated_answer"],
            "label"           : 1,
            "answer_type"     : "hallucinated_answer",
        })
    return instances

train_instances = expand_to_scoring_instances(sampled_data, train_ids)
test_instances  = expand_to_scoring_instances(sampled_data, test_ids)

print("✓ Scoring instances created:")
print(f"  Train: {len(train_instances)} "
      f"({sum(1 for x in train_instances if x['label']==0)} faithful + "
      f"{sum(1 for x in train_instances if x['label']==1)} hallucinated)")
print(f"  Test : {len(test_instances)} "
      f"({sum(1 for x in test_instances if x['label']==0)} faithful + "
      f"{sum(1 for x in test_instances if x['label']==1)} hallucinated)")
print(f"\n--- Sample Scoring Instance ---")
print(json.dumps(train_instances[0], indent=2, ensure_ascii=False))

✓ Scoring instances created:
  Train: 640 (320 faithful + 320 hallucinated)
  Test : 160 (80 faithful + 80 hallucinated)

--- Sample Scoring Instance ---
{
  "instance_id": "0_right",
  "sample_id": 0,
  "original_id": 1824,
  "knowledge": " She is best known as a featured guest vocalist on several \"Billboard\" Hot 100 charting songs, such as G-Eazy's \"Me, Myself & I\", David Guetta's \"Hey Mama\", Martin Garrix's \"In the Name of Love\" and Cash Cash's \"Take Me Home\".\"Hey Mama\" is a song by French DJ and record producer David Guetta, featuring vocals from Nicki Minaj and Bebe Rexha, as well as production from Dutch DJ and producer Afrojack.",
  "question": "Bebe Rexha was a singer who guested on the David Guetta song that was produced by which Dutch DJ?",
  "candidate_answer": "Afrojack",
  "label": 0,
  "answer_type": "right_answer"
}


In [13]:
# ============================================================
# CELL 7: Project summary & GPU verification
# ============================================================
import torch

print("=" * 62)
print("         PROJECT 7 — DATA SETUP COMPLETE")
print("=" * 62)

# GPU Check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU            : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("  GPU            : ⚠ NOT AVAILABLE — switch runtime to GPU!")

print(f"\n  Dataset source : https://github.com/RUCAIBox/HaluEval")
print(f"  Dataset split  : QA subset (Li et al., 2023)")
print(f"  Full dataset   : {len(raw_data)} question-answer pairs")
print(f"  Sampled subset : {len(sampled_data)} pairs (seed={RANDOM_SEED})")
print(f"  Train split    : {len(train_ids)} pairs → {len(train_ids)*2} scoring instances")
print(f"  Test split     : {len(test_ids)} pairs → {len(test_ids)*2} scoring instances")

print(f"\n  Files in Google Drive ({BASE_DIR}):")
print(f"    data/halueval_qa_raw.json      — full {len(raw_data)}-pair dataset")
print(f"    data/halueval_qa_sampled.json  — {len(sampled_data)}-pair fixed sample")
print(f"    data/train_test_split.json     — 80/20 split indices")
print(f"    checkpoints/                   — Task 1 & 2 generation checkpoints")
print(f"    outputs/                       — AUC-PR results & models")
print("=" * 62)
print("\n✓ Data setup complete. Ready to start Task 1, 2, and 3.")

         PROJECT 7 — DATA SETUP COMPLETE
  GPU            : Tesla T4 (14.6 GB VRAM)

  Dataset source : https://github.com/RUCAIBox/HaluEval
  Dataset split  : QA subset (Li et al., 2023)
  Full dataset   : 10000 question-answer pairs
  Sampled subset : 400 pairs (seed=42)
  Train split    : 320 pairs → 640 scoring instances
  Test split     : 80 pairs → 160 scoring instances

  Files in Google Drive (/content/drive/MyDrive/NLP_Assignment_2_Project7):
    data/halueval_qa_raw.json      — full 10000-pair dataset
    data/halueval_qa_sampled.json  — 400-pair fixed sample
    data/train_test_split.json     — 80/20 split indices
    checkpoints/                   — Task 1 & 2 generation checkpoints
    outputs/                       — AUC-PR results & models

✓ Data setup complete. Ready to start Task 1, 2, and 3.
